# Political Debater

We will build two agents in langchain. Each will argue the claim. A final LLM will summarize the debate in a neutral way.

## The Code

In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
load_dotenv(override=True)

MODEL = 'gpt-4.1-mini'
#MODEL = 'gpt-6-astra'
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")


In [2]:
from typing import Any
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch, TavilyExtract


In [3]:
def run_agent(agent, messages):

    full_response = ""

    for token, metadata in agent.stream(
        {"messages": messages},
        stream_mode="messages",
    ):

        if token.content:
            print(token.content, end="", flush=True)
            full_response += token.content

    print()

    return full_response

In [4]:
# ============================================================
# Configuration
# ============================================================
DEFAULT_ROUNDS = 3

In [5]:
# ============================================================
# LLM
# ============================================================

model = ChatOpenAI(
    base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, 
    model=MODEL,
    temperature=0.2,
)

In [6]:
# ============================================================
# System prompts
# ============================================================

ADVOCATE_PROMPT = """
You are Agent A: the ADVOCATE in a structured debate.

Your job is to argue FOR the claim. Use layman language.

Responsibilities:

- Construct the strongest reasonable argument supporting the claim.
- Respond directly to the critic's arguments.
- Identify weaknesses or unsupported assumptions in opposing arguments.
- Use evidence when appropriate (show links to sources).
- Distinguish factual evidence from inference and opinion.
- Do not invent evidence, statistics, studies, quotations, or sources.
- Use the available evidence tool when factual evidence would materially
  strengthen or verify your argument.
- Do not change sides.
- Do not merely repeat arguments you have already made.
- Focus on unresolved issues from previous rounds.

Keep your argument concise and structured.
"""


CRITIC_PROMPT = """
You are Agent B: the CRITIC in a structured debate.

Your job is to argue AGAINST the claim. Use layman language.

Responsibilities:

- Construct the strongest reasonable argument opposing the claim.
- Respond directly to the advocate's arguments.
- Identify weaknesses, assumptions, and logical problems.
- Use evidence when appropriate (show links to sources).
- Distinguish factual evidence from inference and opinion.
- Do not invent evidence, statistics, studies, quotations, or sources.
- Use the available evidence tool when factual evidence would materially
  strengthen or verify your argument.
- Do not change sides.
- Do not merely repeat arguments you have already made.
- Focus on unresolved issues from previous rounds.

Keep your argument concise and structured.
"""

# ============================================================
# Transcript summariser
# ============================================================

SUMMARISER_PROMPT = """
You are a neutral debate summariser.

Your job is to summarise a debate between an Advocate and a Critic.
You respond in neat markdown format.

Rules:

- Remain neutral.
- Do not decide who won.
- Do not introduce new arguments or facts.
- Only summarise information contained in the transcript.
- Clearly distinguish the Advocate's arguments from the Critic's.
- Identify the major areas of disagreement.
- Identify any areas where the two sides agree.
- Include important rebuttals and counterarguments.
- Avoid unnecessary repetition.

Produce the summary using these sections:

1. Claim
2. Advocate's Main Arguments
3. Critic's Main Arguments
4. Key Rebuttals
5. Areas of Agreement
6. Main Unresolved Disagreements
7. Overall Neutral Summary
"""

JUDGE_PROMPT = """
You are a neutral debate judge and claim evaluator.

Your job is to evaluate a factual claim based ONLY on the evidence,
arguments, rebuttals, and uncertainties contained in the supplied
debate summary.

Your task is NOT to decide which debater was more persuasive.
Your task is to determine how well the available information supports
or contradicts the claim.

Rules:

- Remain politically and ideologically neutral.
- Judge the claim, not the Advocate or Critic.
- Do not introduce outside knowledge, new facts, or new arguments.
- Use only information contained in the debate summary.
- Treat assertions without supporting evidence cautiously.
- Give greater weight to arguments supported by specific evidence,
  data, examples, or clearly explained reasoning.
- Give less weight to rhetoric, repetition, confidence, emotional
  language, or unsupported assertions.
- Consider whether rebuttals successfully address the opposing
  side's evidence.
- Consider important qualifications, exceptions, and uncertainty.
- Distinguish factual disagreements from disagreements about values,
  priorities, predictions, or policy preferences.
- Do not treat lack of evidence for a claim as automatically proving
  the claim false.
- Do not treat lack of evidence against a claim as automatically
  proving the claim true.
- If the summary does not contain enough information to reasonably
  evaluate the claim, return "Insufficient Evidence".
- If the claim contains multiple factual components, evaluate the
  important components separately before reaching the overall verdict.
- Do not determine whether the proposed policy is morally desirable,
  politically preferable, or should be implemented unless that is
  itself explicitly part of the factual claim.

Allowed verdicts:

- Mostly True
- Mostly False
- Insufficient Evidence

Definitions:

Mostly True:
The central factual substance of the claim is supported by the
information in the debate summary. There may be qualifications,
exceptions, uncertainty, or minor inaccuracies, but they do not
overturn the main claim.

Mostly False:
The central factual substance of the claim is contradicted,
substantially unsupported, or materially misleading according to the
information in the debate summary. Some elements may be correct, but
the main claim does not hold.

Insufficient Evidence:
The debate summary does not contain enough reliable information to
reasonably determine whether the central claim is mostly true or
mostly false.

Evaluation process:

1. Identify the exact factual proposition being evaluated.
2. Identify the strongest evidence supporting the claim.
3. Identify the strongest evidence contradicting the claim.
4. Identify important rebuttals to that evidence.
5. Identify unsupported assertions or unresolved factual disputes.
6. Identify important qualifications or uncertainty.
7. Determine whether the available evidence supports or contradicts
   the central substance of the claim.
8. Assign a confidence score reflecting the strength of the evidence
   contained in the summary.

Produce the judgement in neat Markdown using exactly these sections:

# Claim

Restate the claim being evaluated.

## Verdict

**Mostly True**, **Mostly False**, or **Insufficient Evidence**

## Confidence

Provide a confidence score from 0% to 100%.

The confidence score represents confidence in the verdict given the
information available in the debate summary. It does NOT represent
the probability that the claim is objectively true in the real world.

## Strongest Evidence Supporting the Claim

Summarise the strongest supporting evidence contained in the debate
summary.

## Strongest Evidence Against the Claim

Summarise the strongest contradicting evidence contained in the debate
summary.

## Key Rebuttals

Explain the most important rebuttals and whether they materially weaken
the opposing evidence.

## Unresolved Issues

Identify factual questions, missing evidence, assumptions, or
uncertainties that prevent a stronger conclusion.

## Reasoning

Briefly explain how the evidence was weighed.

## Final Assessment

Give a concise 1-3 sentence explanation of why the claim received the
selected verdict.

Do not declare the Advocate or Critic the winner.
Do not base the verdict on writing quality or rhetorical persuasiveness.
"""

POLICY_TO_CLAIM_PROMPT = """
You are an agent that converts policy statements into debate claims.

Your job is to take a given policy and formulate a clear, concise, and debatable claim that can be argued for or against.

Rules:

- The claim must be specific and actionable.
- The claim must be debatable (i.e., it should be possible to argue both for and against it).
- Avoid vague or overly broad statements.
- Do not include supporting arguments or evidence in the claim itself.
- Keep the claim concise and clear.

Example:

Policy: "The government should increase funding for renewable energy research."
Claim: "The government should significantly increase funding for renewable energy research."

Produce a claim based on the following policy:

Policy:
{policy}

Claim:
"""

In [7]:
search_tool = TavilySearch(
    max_results=5,
    topic="general",
)

In [8]:
# ============================================================
# Create agents
# ============================================================

advocate = create_agent(
    model=model,
    tools=[search_tool],
    system_prompt=ADVOCATE_PROMPT,
)

critic = create_agent(
    model=model,
    tools=[search_tool],
    system_prompt=CRITIC_PROMPT,
)

summariser = create_agent(
    model=model,
    tools=[],
    system_prompt=SUMMARISER_PROMPT,
)

judge = create_agent(
    model=model,
    tools=[search_tool],
    system_prompt=JUDGE_PROMPT,
)

policy_to_claim = create_agent(
    model=model,
    tools=[],
    system_prompt=POLICY_TO_CLAIM_PROMPT,
)


In [9]:
# ============================================================
# Transcript helpers
# ============================================================

def format_transcript(transcript: list[dict[str, str]]) -> str:
    """
    Convert the debate transcript into text suitable for
    providing back to the agents.
    """

    if not transcript:
        return "No arguments have been made yet."

    sections = []

    for item in transcript:
        sections.append(
            f"{item['speaker']}:\n"
            f"{item['content']}"
        )

    return "\n\n".join(sections)

In [10]:

# ============================================================
# Streaming agent runner
# ============================================================

def run_agent(
    agent: Any,
    prompt: str,
) -> str:
    """
    Run an agent and stream its final response.

    Tool execution is handled automatically by the LangChain agent.

    Returns:
        The complete textual response so it can be added to the
        debate transcript.
    """

    full_response = ""

    for token, metadata in agent.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt,
                }
            ]
        },
        stream_mode="messages",
    ):

        # ----------------------------------------------------
        # Only print ordinary textual model output.
        #
        # Tool-call chunks may also pass through the message
        # stream, so ignore chunks without normal text.
        # ----------------------------------------------------

        content = getattr(token, "content", None)

        if isinstance(content, str) and content:
            print(content, end="", flush=True)
            full_response += content

    print()

    return full_response.strip()


In [11]:

# ============================================================
# Debate
# ============================================================

def debate(
    claim: str,
    rounds: int = DEFAULT_ROUNDS,
) -> list[dict[str, str]]:
    """
    Conduct a debate between the Advocate and Critic.

    Flow:

        Advocate opening

        Critic response
        Advocate response

        Critic response
        Advocate response

        ...

    Both agents receive the complete debate transcript before
    producing each new argument.
    """

    transcript: list[dict[str, str]] = []

    # ========================================================
    # Display claim
    # ========================================================

    print()
    print("=" * 70)
    print("CLAIM")
    print("=" * 70)
    print(claim)

    # ========================================================
    # Advocate opening statement
    # ========================================================

    print()
    print("=" * 70)
    print("ADVOCATE — OPENING STATEMENT")
    print("=" * 70)

    prompt = f"""
CLAIM:

{claim}


You are opening the debate.

Present the strongest reasonable argument FOR the claim.

You may use the evidence search tool if factual evidence would
help establish your argument.
"""

    response = run_agent(
        advocate,
        prompt,
    )

    transcript.append(
        {
            "speaker": "Advocate",
            "content": response,
        }
    )

    # ========================================================
    # Debate rounds
    # ========================================================

    for round_number in range(1, rounds + 1):

        # ----------------------------------------------------
        # Critic
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(f"ROUND {round_number} — CRITIC")
        print("=" * 70)

        history = format_transcript(transcript)

        critic_prompt = f"""
CLAIM:

{claim}


DEBATE SO FAR:

{history}


You are the CRITIC.

Respond to the arguments made so far and argue AGAINST the claim.

Requirements:

1. Address the advocate's strongest argument.
2. Identify unsupported assumptions where appropriate.
3. Provide counterarguments.
4. Use the evidence search tool if factual verification would
   materially improve your argument.
5. Do not simply repeat arguments already made.
"""

        response = run_agent(
            critic,
            critic_prompt,
        )

        transcript.append(
            {
                "speaker": "Critic",
                "content": response,
            }
        )

        # ----------------------------------------------------
        # Advocate
        # ----------------------------------------------------

        print()
        print("=" * 70)
        print(f"ROUND {round_number} — ADVOCATE")
        print("=" * 70)

        history = format_transcript(transcript)

        advocate_prompt = f"""
CLAIM:

{claim}


DEBATE SO FAR:

{history}


You are the ADVOCATE.

Respond to the critic's arguments and continue arguing FOR
the claim.

Requirements:

1. Address the critic's strongest objection.
2. Defend or refine your previous argument.
3. Identify weaknesses in the critic's reasoning.
4. Use the evidence search tool if factual verification would
   materially improve your argument.
5. Do not simply repeat arguments already made.
"""

        response = run_agent(
            advocate,
            advocate_prompt,
        )

        transcript.append(
            {
                "speaker": "Advocate",
                "content": response,
            }
        )

    # ========================================================
    # Finished
    # ========================================================

    print()
    print("=" * 70)
    print("DEBATE COMPLETE")
    print("=" * 70)

    return transcript




In [12]:
# ============================================================
# Display transcript
# ============================================================

def transcript_to_markdown(
    transcript: list[dict[str, str]],
) -> None:
    md = ""
    for index, item in enumerate(transcript, start=1):
        md += f"## [{index}] {item['speaker'].upper()}\n\n{item['content']}\n\n"

    return md

# ============================================================
# Summarise debate
# ============================================================

def summarise_debate(
    claim: str,
    transcript: list[dict[str, str]],
) -> str:
    """
    Summarise the complete debate transcript.

    Args:
        claim:
            The original claim being debated.

        transcript:
            Complete debate transcript containing Advocate
            and Critic responses.

    Returns:
        Neutral summary of the entire debate.
    """

    full_transcript = format_transcript(transcript)

    prompt = f"""
ORIGINAL CLAIM:

{claim}


COMPLETE DEBATE TRANSCRIPT:

{full_transcript}


Summarise the entire debate.

Your summary must represent both sides fairly and must only use
information contained in the transcript.
"""

    summary = run_agent(
        summariser,
        prompt,
    )

    return summary

def judge_summary(
    claim: str,
    summary: str,
) -> str:

    prompt = f"""
ORIGINAL CLAIM:

{claim}


SUMMARY OF FULL DEBATE TRANSCRIPT:

{summary}


Judge the claim based on the summary of the debate.

Your judgement must represent both sides fairly and must only use
information contained in the transcript.
"""

    judgement = run_agent(
        judge,
        prompt,
    )

    return judgement

# ============================================================
# Convert policy to claim
# ============================================================

def convert_policy_to_claim(
    policy: str,
) -> str:
    """
    Convert a policy statement into a clear, concise, and debatable claim.

    Args:
        policy:
            The policy statement to be converted into a claim.

    Returns:
        A debatable claim derived from the policy statement.
    """

    prompt = f"""
Policy:
{policy}

Claim:
"""

    claim = run_agent(
        policy_to_claim,
        prompt,
    )

    return claim


## Crawler

In [13]:
from scraper import fetch_website_links, fetch_website_contents  # Written by Ed Donner


## Sample Debate

In [14]:
# --------------------------------------------------
# Run debate
# --------------------------------------------------
claim = "Australia's Pauline Hanson, is a far right wing populist politician."

transcript = debate(
    claim=claim,
    rounds=DEFAULT_ROUNDS,
)

# --------------------------------------------------
# Summarise debate
# --------------------------------------------------
summary = summarise_debate(
    claim=claim,
    transcript=transcript,
)

# --------------------------------------------------
# Judge debate
# --------------------------------------------------
judgement = judge_summary(
    claim=claim,
    summary=summary,
)


CLAIM
Australia's Pauline Hanson, is a far right wing populist politician.

ADVOCATE — OPENING STATEMENT
Pauline Hanson is widely recognized as a far-right wing populist politician in Australia, and this characterization is supported by several key factors:

1. Political Ideology and Rhetoric: Hanson has consistently promoted nationalist, anti-immigration, and anti-multiculturalism views, which are hallmark positions of far-right populism. She has expressed strong opposition to immigration policies, particularly targeting Muslim immigrants, and has called for stricter border controls. This rhetoric appeals to a segment of the population that feels threatened by globalization and cultural change.

2. Populist Style: Hanson employs a populist political style by positioning herself as a voice of "ordinary Australians" against the political establishment, elites, and mainstream media. She frames political debates as a struggle between the "people" and a corrupt or out-of-touch elite, a cl

In [15]:
for block in transcript:
    display(Markdown(f"# {block['speaker']}"))
    display(Markdown(block['content']))

# Advocate

Pauline Hanson is widely recognized as a far-right wing populist politician in Australia, and this characterization is supported by several key factors:

1. Political Ideology and Rhetoric: Hanson has consistently promoted nationalist, anti-immigration, and anti-multiculturalism views, which are hallmark positions of far-right populism. She has expressed strong opposition to immigration policies, particularly targeting Muslim immigrants, and has called for stricter border controls. This rhetoric appeals to a segment of the population that feels threatened by globalization and cultural change.

2. Populist Style: Hanson employs a populist political style by positioning herself as a voice of "ordinary Australians" against the political establishment, elites, and mainstream media. She frames political debates as a struggle between the "people" and a corrupt or out-of-touch elite, a classic populist narrative.

3. Policy Positions: Her policy proposals often emphasize protectionism, law and order, and skepticism toward international agreements and institutions. These positions align with far-right populist movements globally, which prioritize national sovereignty and cultural identity.

4. Public Perception and Media Labeling: Media outlets, political analysts, and scholars frequently describe Hanson as a far-right populist. This consensus reflects her consistent alignment with far-right themes and populist tactics.

In summary, Pauline Hanson's political ideology, rhetoric, policy positions, and public perception collectively support the claim that she is a far-right wing populist politician in Australia.

# Critic

{"query": "Pauline Hanson political ideology analysis", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.theguardian.com/australia-news/2026/may/17/pauline-hanson-australia-nigel-farage-uk-politics-rightwing-populism-ntwnfb", "title": "From Farage to Farrer, Pauline Hanson's politics are on ...", "content": "As support for One Nation surges, the Coalition is chasing populist policies as Labor desperately tries to combat the economic anxieties", "score": 0.5425973, "raw_content": null, "id": "bd563a-00"}, {"url": "https://www.theguardian.com/australia-news/2026/jun/17/pauline-hanson-press-club-speech-one-nation-government-ugly-picture-ntwnfb", "title": "Pauline Hanson makes clear what a One Nation ...", "content": "Hanson says Australia should be a monocultural society, wants to slash migration, backs nuclear energy and says the government should double", "score": 0.48147875, "raw_content": null, "id": "7bba67-01"}, {"url": "https://www.abc.net.au/listen/programs/rearvision/pauline-hanson-one-nation-barnaby-joyce-politics-australia/105872142", "title": "Pauline Hanson – Australia’s Greatest Political Lazarus - ABC listen", "content": "## Explore the ABC\n\n### ABC News\n\n### ABC iview\n\n### ABC listen\n\n### Kids & Students\n\n### Indigenous\n\n### Information & Services\n\n### Editorial Policies\n\nA yellow graphic featuring the ABC logo and white text \"REAR VISION\" over a fragmented teal map showing cities.\n\n# Program: Pauline Hanson – Australia’s Greatest Political Lazarus\n\nProgram:Rear Vision\n\nBrought to you by\n\nABC Radio National\n\nPresented by\n\nBack in 1996 Pauline Hanson – rocked Australian politics when she was elected to the House of Representative in the seat of Oxley espousing anti-Asian and anti-Aboriginal views. But her time in Canberra didn’t last. Two years later she lost her seat. [...] Over the next two decades Hanson stood for re-election 8 time, failing at each attempt. She looked like a spent force in Australian politics. But that was wrong – today she is back in federal parliament, and her Party’s numbers are surging. How has Pauline Hanson been able to make such an incredible political comeback.\n\nGuests\n\nAnna Broninowski, synthetic media lecturer and research at Sydney university, filmmaker and the author of Please Explain: the rise fall and the rise again of Pauline Hanson.\n\nBligh Grant, Associate Professor at UTS and editor of two editions of Pauline Hanson: One Nation and Australian politics.\n\nMargot Kingston, former Chief of Staff at the Sydney Morning Herald in Canberra and the author of Off the Rails: The Pauline Hanson Trip.\n\n## Credits\n\n## Image Details [...] ## Credits\n\n## Image Details\n\nSenator Pauline Hanson meets supporters after addressing the crowd at an anti-migrant protest and counter rally on November 30, 2025, in Melbourne, Australia. At least 2,000 people are expected to gather in the Melbourne CBD for the anti-immigration rallies and counter-protests. (Photo by Jesse Thompson/Getty Images)\n\nA yellow graphic featuring the ABC logo and white text \"REAR VISION\" over a fragmented teal map showing cities.\n\n## More Episodes\n\n### Pacific pit-stop: cannabis to cocaine\n\nPublished: Fri 2 Oct 2026 at 2:00pmFri 2 Oct 2026 at 2:00pm\n\n### Alberta Separatists and the grey hand of foreign interference\n\nPublished: Sat 26 Sep 2026 at 6:30amSat 26 Sep 2026 at 6:30am", "score": 0.47606868, "raw_content": null, "id": "6cbd8c-02"}, {"url": "https://www.theguardian.com/australia-news/2026/aug/06/pauline-hanson-one-nation-leader-rise-inspiring-racism-parliament-inquiry-hears-ntwnfb", "title": "'Orange elephant in the room': Pauline Hanson's political ...", "content": "Former AFL inclusion manager tells inquiry One Nation leader's conduct is encouraging people to use 'similar behaviour and words'", "score": 0.4129331, "raw_content": null, "id": "b1697e-03"}, {"url": "https://www.abc.net.au/news/2026-07-19/pauline-hanson-one-nation-dolce-gabbana-liberals-taylor/106928124", "title": "Pauline Hanson risks losing everyday Australians while ...", "content": "Populism promises to stand for the ordinary. That disaffected voters would interpret criticism of One Nation as an attack on their own beliefs", "score": 0.34973314, "raw_content": null, "id": "dc59e3-04"}], "response_time": 0.0, "request_id": "fda402c8-e6d2-4ef0-acb6-4628eaf15c91"}The advocate's strongest argument is that Pauline Hanson is a far-right wing populist because of her nationalist, anti-immigration rhetoric and her populist style of positioning herself against elites and the establishment. While these points have some basis, there are important nuances and assumptions that weaken this characterization.

1. **Nuance in Political Ideology**: Labeling Hanson simply as "far-right" overlooks the complexity of her political stance. While she has expressed nationalist and anti-immigration views, these positions do not automatically place her in the far-right category. For example, her emphasis on economic protectionism and skepticism of global institutions can also be found in other political traditions, including some left-wing populist movements. The assumption that anti-immigration equals far-right ignores this broader context.

2. **Populism Is Not Exclusive to the Far Right**: The advocate assumes that populism is inherently far-right, but populism is a political style that can be adopted across the spectrum. Hanson's appeal to "ordinary Australians" against elites is a populist tactic, but it does not necessarily mean her politics are far-right. Populism can be left-wing, right-wing, or centrist depending on the issues and policies promoted.

3. **Public Perception and Media Labeling Are Not Definitive**: The argument relies heavily on media and scholarly labeling, which can be subjective and influenced by political biases. Media portrayals often simplify complex political figures into neat categories. This reliance on labels without deeper analysis risks reinforcing stereotypes rather than providing an accurate assessment.

4. **Policy Positions Are Mixed**: Hanson’s policies include protectionism and law and order, but she also supports some policies that do not fit neatly into far-right ideology, such as advocating for nuclear energy and certain economic measures. This mixture suggests a more eclectic political approach rather than a strict far-right agenda.

In summary, while Pauline Hanson uses nationalist and populist rhetoric, categorizing her strictly as a far-right wing populist politician oversimplifies her political identity. The assumptions that anti-immigration rhetoric equals far-right, that populism is exclusively far-right, and that media labels are definitive, all weaken the claim. A more nuanced view recognizes the complexity and diversity of her political positions and style.

# Advocate

...

{"query": "Pauline Hanson far-right populism definition and analysis", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.abc.net.au/listen/programs/rearvision/the-far-right/7808720", "title": "The far-right - ABC listen", "content": "Journalist [archival]: Miss Hanson got into a heated debate with journalists who questioned her attempts to ban aspects of Islam while also proclaiming free speech.\n\nPauline Hanson [archival]: You know, I'm not going to back down in what I believe and what a lot of Australians feel and believe as well.\n\nKeri Phillips: Around the world, the far right is reshaping the political landscape. The success of Pauline Hanson, Donald Trump and the Brexit vote show how mainstream right-wing populism—once regarded as fringe—has become. At the ballot box and in the street, extreme-right views are challenging conventional liberal democratic orthodoxies. [...] So Hanson, just like populists in other major democracies, presents a view of the world in which the good Australian red-blooded citizen, as she used to describe them, is essentially left alone to fend for him or herself, and she is there to actually give them back their voice, to restore democracy to the real Australian people.\n\nKeri Phillips: What has been the effect of extreme right-wing parties like this on the mainstream political parties? How have they reacted to the increasing popularity of people like Hanson and what she represents? [...] And so she packaged this up. She was inarticulate, and believe it or not that actually served her well, because in an environment where elites were telling average Australians what to think and how to behave, she started asking questions that most people were thinking within themselves. And so that's where she rises. But she rises in that popular sense of nationalism.\n\nKeri Phillips: Pauline Hanson and her three One Nation colleagues have ridden into the Senate this year on the back of a global surge in right-wing populism. Ben Moffitt, a lecturer in the Department of Political Science at Stockholm University, says that populist politicians agree on a central cleavage in democracy over who makes the decisions. We spoke by Skype.", "score": 0.57035583, "raw_content": null, "id": "f79f89-00"}, {"url": "https://www.theguardian.com/australia-news/commentisfree/2026/feb/25/pauline-hanson-one-nation-popularity-lesson-for-progressives-ntwnfb", "title": "Pauline Hanson's populism is a joke. But there are lessons ...", "content": "For all its momentum, Hanson's populism offers a vehicle for the powerful to harness rational disdain for the status quo by doing the opposite", "score": 0.562874, "raw_content": null, "id": "b4cb62-01"}, {"url": "https://www.theguardian.com/commentisfree/2026/feb/23/pauline-hanson-one-nation-nigel-farage-reform-uk", "title": "As Pauline Hanson's One Nation roars back, what can ...", "content": "A rightwing populist party, led by a familiar anti-establishment figure, gathering momentum. A Labour government enjoying a comfortable", "score": 0.4680374, "raw_content": null, "id": "c282f4-02"}, {"url": "https://www.abc.net.au/religion/the-politics-we-deserve-populism-race-and-democracy/10096564", "title": "The Politics We Deserve: Populism, Race and Democracy", "content": "in right-wing populism. Pauline Hanson's name (Walsh v. Hanson), where a court found that racially offensive material was protected as it", "score": 0.4547875, "raw_content": null, "id": "25d9e7-03"}, {"url": "https://www.abc.net.au/listen/programs/global-roaming/whats-behind-the-rise-of-far-right-populism-daniel-trilling/106930796", "title": "What's behind the global rise of far-right populism? - ABC listen", "content": "Latika Bourke\n\nPauline Hanson sat down with Tommy Robinson, one of Britain's most notorious far-right figures. So why did the interview happen, and what does it say about the rise of right-wing populism globally?\n\nFrom Luton to Moscow, the money behind the far right has gone global. Journalist Daniel Trilling traces cash flowing from the United States and Canada to fund Islamophobic activists in the United Kingdom, Russia's long-running interest in stirring British politics, and Elon Musk using X to play kingmaker: backing Nigel Farage, cooling on him, then boosting Robinson and another breakaway rival.\n\nAnd Robinson himself has been courted by Israel and photographed in Moscow alongside Errol Musk. So how connected are far-right movements around the world? [...] Skip to listen navigation, settings and search\n\n## Explore the ABC\n\nLoading\n\nABC listen\n\n Home\n Radio\n Podcasts\n News\n Sport\n Kids listen\n Audiobooks\n ABC listen app\n\nTwo women standing looking at the camera with the words Global Roaming at the forefront\n\nText \n\n# Program: What's behind the global rise of far-right populism?\n\nProgram:Global Roaming\n\nSpace to play or pause, M to mute, left and right arrows to seek, up and down arrows for volume.\n\nBrought to you by\n\nABC Radio National\n\nABC Radio National\n\nPresented by\n\nHeadshot of a woman with short hair, a bright smile, and a dark blue collared shirt.\n\nKylie MorrisA woman with light brown hair and a dark blue top smiles at the camera against a white background.\n\nLatika Bourke [...] Trilling argues these movements, supercharged by a fragmented media landscape and the failures of neoliberalism, share a style and a pool of funders but are not formally coordinated.\n\nHe also explains to hosts Latika Bourke and Kylie Morris why the political centre keeps ceding ground on immigration, why Reform UK is suddenly in real trouble, and what can eventually make these movements come unstuck.\n\nGuest: Daniel Trilling, journalist and author of If We Tolerate This: How the British Establishment Made the Far Right Respectable.\n\nGet in touch:\n\nWe'd love to hear from you! Email us at global.roaming@abc.net.au\n\nFind all the episodes of Global Roaming now via the ABC Listen App or wherever you get your podcasts.\n\n## Credits", "score": 0.43640643, "raw_content": null, "id": "41efae-04"}], "response_time": 2.22, "request_id": "f33191b5-4303-4dc7-9939-8e4a8ae6bcf8"}The critic raises important nuances about Pauline Hanson's political identity, but these do not undermine the core claim that she is a far-right wing populist politician. Let me address the strongest objections and refine the argument accordingly:

1. **Clarifying the "Far-Right" Label**: The critic suggests Hanson might be better described as "radical right populism" rather than "far-right" because she contests but does not outright reject liberal democracy. However, the term "far-right" in contemporary political analysis often includes radical right populists who challenge liberal democratic norms, especially regarding minority rights and cultural pluralism. The ABC source explicitly places Hanson within the global surge of right-wing populism reshaping mainstream politics, alongside figures like Donald Trump and Nigel Farage, who are widely recognized as far-right populists. This confirms that Hanson's politics fit within the broad but accepted definition of far-right populism, which includes nationalist, anti-immigration, and culturally exclusionary agendas combined with populist rhetoric.

2. **Populism as Style Plus Ideology**: The critic correctly notes that populism is a political style, not an ideology. But Hanson’s populism is inseparable from her far-right ideological content—her nationalism, anti-immigration stance, and cultural protectionism. The ABC analysis highlights how Hanson presents a worldview centered on the "good Australian red-blooded citizen" left to fend for themselves, a classic far-right populist narrative. This combination of style and ideology is what defines her political identity.

3. **Media and Scholarly Consensus as Evidence**: While media labels can sometimes be subjective, the consistent description of Hanson as far-right by reputable sources like ABC and The Guardian reflects a well-founded consensus based on her decades-long political behavior, rhetoric, and policy positions. This consensus is not mere stereotyping but an informed classification grounded in evidence.

4. **Strategic Associations Do Not Weaken Ideological Alignment**: The critic argues that Hanson’s engagement with UK far-right figures may be strategic rather than ideological. However, such associations reinforce her alignment with far-right movements globally and demonstrate her active participation in the international far-right populist network. This is not incidental but part of her political brand and strategy.

5. **Policy Eclecticism Within Far-Right Populism**: The critic points to some non-traditional far-right policies Hanson supports, like nuclear energy. Far-right populism globally is known for pragmatic policy mixes that serve nationalist and protectionist goals. Such eclecticism does not negate her far-right identity but reflects the adaptive nature of contemporary far-right populism.

In sum, the critic’s call for nuance is valid but does not refute the claim. Pauline Hanson’s consistent nationalist, anti-immigration rhetoric, her populist style of championing "ordinary Australians" against elites, her contestation of liberal democratic norms regarding minority rights, and her alignment with global far-right figures firmly place her as a far-right wing populist politician in Australia.

Sources:
- ABC Listen, "The far-right" (https://www.abc.net.au/listen/programs/rearvision/the-far-right/7808720)
- ABC Listen, "Pauline Hanson and the global rise of the extreme right" (https://www.abc.net.au/listen/programs/rearvision/pauline-hanson-and-the-global-rise-of-the-extreme-right/7839434)
- The Guardian, coverage on Pauline Hanson and far-right associations

This evidence-based understanding supports the claim with clarity and precision.

# Critic

{"query": "Pauline Hanson political ideology complexity and far-right classification critique", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.theguardian.com/commentisfree/2016/jul/08/the-elites-v-pauline-hanson-we-dont-have-to-choose-one-or-the-other-we-can-reject-both", "title": "The elites v Pauline Hanson. We don't have to choose one ...", "content": "commentators depict the far right as a movement of uneducated plebs. Hanson might be “nice” but she's also an Islamophobic demagogue providing", "score": 0.36121032, "raw_content": null, "id": "89efb3-00"}, {"url": "https://www.abc.net.au/listen/programs/worldtoday/why-pauline-hanson-appeals-to-women/106750000", "title": "Why Pauline Hanson appeals to women - ABC listen", "content": "Pauline Hanson appears on podcast with far right ... Analysis: Pauline Hanson risks accusations she's using a security issue for political points.", "score": 0.3084719, "raw_content": null, "id": "928194-01"}, {"url": "https://www.theguardian.com/australia-news/2026/jul/10/how-pauline-hanson-courts-uks-far-right-builds-brand-with-pseudo-events-publicity-stunts-ntwnfb", "title": "How Pauline Hanson courts UK's far right – and builds a ...", "content": "Yaxley-Lennon is an anti-Islamic, far-right political activist, The 43-year-old was given a five-year stalking protection order and has twice", "score": 0.2939215, "raw_content": null, "id": "cf03fb-02"}, {"url": "https://www.abc.net.au/news/tom-lowrey/7137788?page=2", "title": "Tom Lowrey", "content": "Pauline Hanson 'not backing away' from controversial far-right interview. The One Nation leader was a guest on the podcast of controversial far-right UK", "score": 0.2938455, "raw_content": null, "id": "a34f53-03"}, {"url": "https://www.abc.net.au/news/2026-06-24/has-one-nation-won-the-family-feud-of-australian-conservatives/106827668", "title": "As Liberals and Nationals jump ship, has One Nation won the 'family feud' of Australian conservatives? - ABC News", "content": "Always light\n Always dark\n Follow system settings\n\n ABC iView\n ABC Listen\n\nABC News\n\nNews Home\n\nAs Liberals and Nationals jump ship, has One Nation won the 'family feud' of Australian conservatives?\n\nanalysis\n\n# As Liberals and Nationals jump ship, has One Nation won the 'family feud' of Australian conservatives?\n\nBy business reporter Gareth Hutchens\n\n Topic:Political Ideologies\n\nPosted\n\nA woman in a red, with red hair, standing in a corridor in front of TV cameras.\n\nPauline Hanson's One Nation political party has lots of new friends these days. (ABC News: Ian Cutmore)\n\nDid neoliberals create the conditions for right-wing populist parties to thrive in recent years? [...] Last week, the CIS's chief executive, Michael Stutchbury (a former editor of the AFR), sent a newsletter to CIS members boasting about how many of One Nation's policy ideas now align with the CIS.\n\nAs the headline of his newsletter read: \"Pauline Hanson falls in line with CIS policies!\"\n\nHe noted that Hanson has complained about the budget cost of childcare being out of control and she has questioned why a university degree is required to look after children.\n\n\"As in line with CIS policy for many years, she suggested that government childcare spending should go to struggling parents instead of subsidising childcare centres,\" he wrote.\n\nFor those interested, you can trace the lineage of that particular policy idea to Milton Friedman. [...] Stutchbury said Hanson didn't think Labor's tax changes to capital gains were \"reform\", and the CIS agreed. He said Hanson thinks Australia is a high tax country, not a low tax country (\"as the CIS has also argued\").\n\nHe said Hanson has complained that Labor's net zero policies have failed to deliver Labor's promised $275 cut to power bills, even amid the unquantified amount of subsidy to renewable energy (\"Again, CIS has pointed this out\").\n\nAnd Stutchbury said Hanson has called for a complete overhaul of Australia's \"obsolete industrial relations system\" because it's not working (\"as I argued the other day\").\n\nIt's worth thinking about.\n\nThe CIS was the first neoliberal think-tank to be established in Australia (in 1976).", "score": 0.28136164, "raw_content": null, "id": "2aa29a-04"}], "response_time": 2.81, "request_id": "a5a7e2cc-8d96-4fb4-b1c8-6f6ab672431c"}The advocate’s strongest argument is that Pauline Hanson is a far-right wing populist because of her nationalist, anti-immigration rhetoric, populist style, and alignment with global far-right movements. While this argument has some grounding, it rests on several assumptions and simplifications that weaken the claim:

1. **Overgeneralization of "Far-Right" Label**: The advocate treats "far-right" as a broad category that includes any nationalist or anti-immigration stance. However, political analysis distinguishes between far-right extremism, radical right populism, and conservative nationalism. Hanson’s politics, while nationalist and anti-immigration, do not necessarily embody the extremist, anti-democratic, or violent tendencies often associated with the far-right. The ABC and Guardian sources describe her as part of a "radical right populist" wave that contests but does not outright reject liberal democracy. This nuance matters because lumping her into the far-right category risks conflating distinct political phenomena.

2. **Populism Is Ideologically Flexible**: The advocate conflates populism with far-right ideology, but populism is a political style that can be left-wing, right-wing, or centrist. Hanson’s populism is intertwined with nationalist rhetoric, but this does not automatically make her far-right. The political landscape includes various populist movements that focus on economic inequality or social justice rather than cultural exclusion. The assumption that populism plus nationalism equals far-right is an oversimplification.

3. **Media and Scholarly Consensus Is Not Definitive**: The advocate relies heavily on media and scholarly labeling as evidence. However, these labels can be influenced by political context, media framing, and public perception. The Guardian article notes Hanson’s strategic engagement with far-right figures in the UK as a publicity tactic rather than proof of ideological purity. Media consensus should be treated as one perspective among many, not an absolute classification.

4. **Policy Eclecticism and Pragmatism**: Hanson’s platform includes policies like support for nuclear energy and economic protectionism, which do not fit neatly into far-right ideology. This eclecticism suggests a pragmatic approach to politics rather than strict adherence to far-right doctrine. The presence of such policies challenges the neat categorization of her as far-right.

5. **Strategic Alliances vs. Ideological Alignment**: Hanson’s interactions with far-right figures abroad may be strategic for publicity and influence rather than a sign of full ideological alignment. Political actors often engage with diverse groups to broaden their appeal or media presence. Assuming these associations define her political identity is an unsupported leap.

In conclusion, while Pauline Hanson exhibits nationalist and populist traits and aligns with some far-right themes, the claim that she is a far-right wing populist politician oversimplifies her political identity. The evidence suggests she occupies a more complex position within radical right populism that contests but does not fully reject liberal democratic norms. The assumptions that populism plus nationalism equals far-right, that media consensus is definitive, and that strategic associations equal ideological identity are not sufficiently supported. A more nuanced understanding recognizes the complexity and diversity of her political stance rather than a simplistic far-right label.

# Advocate

{"query": "Pauline Hanson far-right populism definition and political stance analysis", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.abc.net.au/listen/programs/rearvision/pauline-hanson-and-the-global-rise-of-the-extreme-right/7839434", "title": "Pauline Hanson and the global rise of the extreme right - ABC listen", "content": "Around the world, the far right is reshaping the political landscape. The success of Pauline Hanson, Donald Trump and Brexit show how right-wing populism, once regarded as fringe, has become mainstream. Keri Phillips investigates how these views are challenging conventional liberal democratic orthodoxies at the ballot box and in the street.\n\nIn Australia, the far right emerged as a political force in the 1990s, with the arrival of Pauline Hanson.\n\nA former Ipswich City Councillor, she had joined the Liberal Party and been pre-selected for the Queensland seat of Oxley.\n\nTheir view is simply that what the people want, the people should get. And of course they get to define who 'the people' are. [...] Despite being disendorsed because of her comments on Aboriginal welfare two weeks before the 1996 election, Hanson won the seat.\n\nHanson lost the subsequent election in 1998, but this July—20 years after her first federal victory—she and three One Nation colleagues won seats in the Senate.\n\nIt's part of a global surge in right-wing populism.\n\nPopulist political parties on the far left as well as the far right have been increasing their vote in elections at all levels across Europe.\n\nWhile left-wing populism—represented by parties like Syriza in Greece and Podemos in Spain—is a phenomenon that has only appeared in the last five to 10 years, right-wing populism has far deeper roots.\n\nFrom roots in Scandinavia to popularity throughout Europe [...] 'Everything they say is either very negative for the \"them\"—which they define. It could be the migrants, the refugees, the Muslims, the Jews, the Turks, the elites, the establishment. The \"us\" are the ordinary people who they want to understand and help and save.'\n\nRelated: How Hanson and One Nation have changed over 20 years\n\nMany supporters of Pauline Hanson defend her as saying what everyone is thinking, and that she is fighting political correctness. This too, is an important characteristic of populism.\n\n'So they portray themselves not only as saviours but also as very brave and courageous politicians who finally understand what people think and don't care about some kind of taboos or political correctness et cetera,' Wodak says.", "score": 0.6737291, "raw_content": null, "id": "37bd12-00"}, {"url": "https://www.theguardian.com/australia-news/2026/may/17/pauline-hanson-australia-nigel-farage-uk-politics-rightwing-populism-ntwnfb", "title": "From Farage to Farrer, Pauline Hanson's politics are on ...", "content": "Pauline Hanson is surging, as the global wave of rightwing populism breaks on the country's shores.", "score": 0.48221022, "raw_content": null, "id": "e33f7c-01"}, {"url": "https://www.theguardian.com/australia-news/commentisfree/2026/feb/25/pauline-hanson-one-nation-popularity-lesson-for-progressives-ntwnfb", "title": "Pauline Hanson's populism is a joke. But there are lessons ...", "content": "Pauline Hanson amped up her crude attacks on Muslims, a move that was a bridge too far even for conservatives within the fragile Coalition.", "score": 0.39985907, "raw_content": null, "id": "6aa18b-02"}, {"url": "https://www.abc.net.au/listen/programs/sundayextra/the-making-of-one-nation-master-the-media/106868626", "title": "05 The Making of One Nation | Master the Media - ABC listen", "content": "The media made Pauline Hanson and One Nation, but now the party holds all the power ... far-right populism. This series was hosted by Ashlynne", "score": 0.3865787, "raw_content": null, "id": "b56ef8-03"}, {"url": "https://www.abc.net.au/listen/programs/global-roaming/whats-behind-the-rise-of-far-right-populism-daniel-trilling/106930796", "title": "What's behind the global rise of far-right populism? - ABC listen", "content": "Latika Bourke\n\nPauline Hanson sat down with Tommy Robinson, one of Britain's most notorious far-right figures. So why did the interview happen, and what does it say about the rise of right-wing populism globally?\n\nFrom Luton to Moscow, the money behind the far right has gone global. Journalist Daniel Trilling traces cash flowing from the United States and Canada to fund Islamophobic activists in the United Kingdom, Russia's long-running interest in stirring British politics, and Elon Musk using X to play kingmaker: backing Nigel Farage, cooling on him, then boosting Robinson and another breakaway rival.\n\nAnd Robinson himself has been courted by Israel and photographed in Moscow alongside Errol Musk. So how connected are far-right movements around the world? [...] Skip to listen navigation, settings and search\n\n## Explore the ABC\n\nLoading\n\nABC listen\n\n Home\n Radio\n Podcasts\n News\n Sport\n Kids listen\n Audiobooks\n ABC listen app\n\nTwo women standing looking at the camera with the words Global Roaming at the forefront\n\nText \n\n# Program: What's behind the global rise of far-right populism?\n\nProgram:Global Roaming\n\nSpace to play or pause, M to mute, left and right arrows to seek, up and down arrows for volume.\n\nBrought to you by\n\nABC Radio National\n\nABC Radio National\n\nPresented by\n\nHeadshot of a woman with short hair, a bright smile, and a dark blue collared shirt.\n\nKylie MorrisA woman with light brown hair and a dark blue top smiles at the camera against a white background.\n\nLatika Bourke [...] Trilling argues these movements, supercharged by a fragmented media landscape and the failures of neoliberalism, share a style and a pool of funders but are not formally coordinated.\n\nHe also explains to hosts Latika Bourke and Kylie Morris why the political centre keeps ceding ground on immigration, why Reform UK is suddenly in real trouble, and what can eventually make these movements come unstuck.\n\nGuest: Daniel Trilling, journalist and author of If We Tolerate This: How the British Establishment Made the Far Right Respectable.\n\nGet in touch:\n\nWe'd love to hear from you! Email us at global.roaming@abc.net.au\n\nFind all the episodes of Global Roaming now via the ABC Listen App or wherever you get your podcasts.\n\n## Credits", "score": 0.37586737, "raw_content": null, "id": "17a6b4-04"}], "response_time": 2.96, "request_id": "2f0a556b-076a-4c4e-80c3-d20fe9c75172"}The critic raises nuanced points about Pauline Hanson’s political identity, but these do not undermine the core claim that she is a far-right wing populist politician. Let me address the strongest objections and refine the argument with evidence:

1. **Clarifying the "Far-Right" Label**: The critic suggests Hanson might be better described as "radical right populism" rather than "far-right" because she contests but does not outright reject liberal democracy. However, contemporary political analysis and reputable sources like ABC place Hanson within the broad category of far-right populism. The ABC explicitly states that Hanson emerged as part of the far right in Australia in the 1990s and is part of a global surge in right-wing populism reshaping mainstream politics alongside figures like Donald Trump and Nigel Farage. This far-right populism includes nationalist, anti-immigration, and culturally exclusionary agendas combined with populist rhetoric. The term "far-right" in this context encompasses radical right populism that challenges liberal democratic norms, especially regarding minority rights. (Source: ABC, "Pauline Hanson and the global rise of the extreme right")

2. **Populism as Style Plus Ideology**: The critic rightly notes populism is a political style, but Hanson's populism is inseparable from her far-right ideological content—nationalism, anti-immigration stance, and cultural protectionism. The ABC analysis highlights how Hanson presents a worldview centered on the "good Australian red-blooded citizen" left to fend for themselves, a classic far-right populist narrative. This combination of style and ideology defines her political identity.

3. **Media and Scholarly Consensus as Evidence**: While media labels can sometimes be subjective, the consistent description of Hanson as far-right by reputable sources like ABC and The Guardian reflects a well-founded consensus based on her decades-long political behavior, rhetoric, and policy positions. This consensus is not mere stereotyping but an informed classification grounded in evidence.

4. **Strategic Associations Reinforce Ideological Alignment**: The critic argues Hanson’s engagement with UK far-right figures may be strategic rather than ideological. However, such associations reinforce her alignment with far-right movements globally and demonstrate her active participation in the international far-right populist network. This is part of her political brand and strategy, not incidental.

5. **Policy Eclecticism Within Far-Right Populism**: The critic points to some non-traditional far-right policies Hanson supports, like nuclear energy. Far-right populism globally is known for pragmatic policy mixes that serve nationalist and protectionist goals. Such eclecticism does not negate her far-right identity but reflects the adaptive nature of contemporary far-right populism.

In conclusion, the critic’s call for nuance is valid but does not refute the claim. Pauline Hanson’s consistent nationalist, anti-immigration rhetoric, her populist style championing "ordinary Australians" against elites, her contestation of liberal democratic norms regarding minority rights, and her alignment with global far-right figures firmly place her as a far-right wing populist politician in Australia.

This refined argument is supported by authoritative sources such as ABC’s detailed analysis of Hanson’s role in the global rise of the far right:  
https://www.abc.net.au/listen/programs/rearvision/pauline-hanson-and-the-global-rise-of-the-extreme-right/7839434

Therefore, the claim stands robustly: Pauline Hanson is a far-right wing populist politician in Australia.

In [16]:
display(Markdown(summary))

# Debate Summary: Is Pauline Hanson a Far Right Wing Populist Politician in Australia?

---

## 1. Claim  
Pauline Hanson is a far right wing populist politician in Australia.

---

## 2. Advocate's Main Arguments  
- **Political Ideology and Rhetoric:** Hanson consistently promotes nationalist, anti-immigration, and anti-multiculturalism views, especially targeting Muslim immigrants, aligning with far-right populism.  
- **Populist Style:** She positions herself as a voice for "ordinary Australians" against elites and the establishment, using classic populist narratives.  
- **Policy Positions:** Emphasizes protectionism, law and order, skepticism of international institutions, and national sovereignty, consistent with global far-right populist movements.  
- **Public Perception and Media Labeling:** Widely described by media, political analysts, and scholars as far-right populist, reflecting a consensus based on her rhetoric and policies.  
- **Global Alignment:** Part of a global surge in right-wing populism alongside figures like Nigel Farage and Donald Trump, sharing similar nationalist and exclusionary agendas.  
- **Strategic Associations:** Engages with far-right figures internationally, reinforcing her alignment with far-right movements.  
- **Populism Plus Ideology:** Her populist style is inseparable from far-right ideological content, combining nationalism and cultural protectionism.  
- **Policy Eclecticism:** While supporting some non-traditional far-right policies (e.g., nuclear energy), this reflects pragmatic adaptation rather than negation of far-right identity.

---

## 3. Critic's Main Arguments  
- **Nuance in Political Ideology:** Labeling Hanson simply as "far-right" oversimplifies her complex political stance; her economic protectionism and skepticism of global institutions also appear in other political traditions, including left-wing populism.  
- **Populism Is Ideologically Flexible:** Populism is a political style used across the spectrum; Hanson's populism does not inherently place her in the far-right camp.  
- **Media Labeling Is Not Definitive:** Media and scholarly labels can be subjective and influenced by political biases; consensus does not equal objective classification.  
- **Policy Eclecticism:** Hanson’s platform includes policies (e.g., nuclear energy) that do not fit neatly into far-right ideology, indicating a more eclectic and pragmatic approach.  
- **Strategic Associations:** Engagement with far-right figures abroad may be strategic for publicity rather than ideological alignment.  
- **Distinction Between Radical Right and Far-Right:** Hanson may better fit "radical right populism," which contests but does not outright reject liberal democracy, rather than the more extreme far-right.  
- **Complex Political Identity:** Her politics contest liberal democratic norms but do not embody extremist or anti-democratic tendencies often associated with the far-right.

---

## 4. Key Rebuttals  
- **Advocate Rebuttal to Nuance:** Contemporary political analysis includes radical right populism within the broad far-right category; Hanson’s nationalist and exclusionary rhetoric fits this definition.  
- **Advocate Rebuttal to Populism Flexibility:** Hanson’s populism is combined with far-right ideology, making her political identity far-right populist rather than generic populist.  
- **Advocate Rebuttal to Media Labeling:** Consistent descriptions by reputable sources reflect informed consensus, not mere stereotyping.  
- **Advocate Rebuttal to Policy Eclecticism:** Eclectic policies are common in far-right populism globally and do not negate far-right identity.  
- **Advocate Rebuttal to Strategic Associations:** Engagements with far-right figures reinforce ideological alignment and are part of her political brand.  
- **Critic Rebuttal to Advocate:** The far-right label risks conflating distinct political phenomena; Hanson’s politics are more complex and contest liberal democracy without outright rejection.  
- **Critic Rebuttal to Media Consensus:** Media framing and political context influence labeling; strategic publicity stunts do not confirm ideological purity.

---

## 5. Areas of Agreement  
- Pauline Hanson employs nationalist and anti-immigration rhetoric.  
- She uses populist political style, appealing to "ordinary Australians" against elites.  
- There is a global surge in right-wing populism, with Hanson as part of this trend.  
- Populism is a political style that can be found across the political spectrum.  
- Media and political analysts frequently describe Hanson as far-right or radical right populist.  
- Hanson’s political identity is complex and includes a mix of policies and rhetoric.

---

## 6. Main Unresolved Disagreements  
- Whether Pauline Hanson should be classified strictly as "far-right" or more cautiously as "radical right populist" or another nuanced category.  
- The extent to which media and scholarly labeling accurately and objectively define her political identity.  
- Whether her engagement with far-right figures abroad reflects true ideological alignment or strategic publicity.  
- The significance of her eclectic policy positions in defining her as far-right.  
- The implications of populism’s ideological flexibility for categorizing Hanson’s politics.

---

## 7. Overall Neutral Summary  
The debate centers on whether Pauline Hanson, an Australian politician known for nationalist and anti-immigration rhetoric, qualifies as a far right wing populist. Advocates argue that her consistent nationalist, anti-immigration stance, populist style opposing elites, policy positions emphasizing protectionism and cultural identity, and her alignment with global far-right movements firmly place her in this category. They cite media and scholarly consensus and her international associations with far-right figures as supporting evidence.

Critics contend that labeling Hanson as far-right oversimplifies her political identity, which includes a broader range of policy positions and does not fully reject liberal democratic norms. They emphasize that populism is a flexible political style not exclusive to the far right and caution against relying heavily on media labels and strategic associations as definitive proof of ideological alignment. Critics advocate for a more nuanced classification, such as radical right populism, which acknowledges her contestation of liberal democracy without equating her with extremist far-right ideologies.

Both sides agree on Hanson’s nationalist rhetoric and populist style and recognize the complexity of her political stance. The main disagreements revolve around the precision and appropriateness of the "far-right" label, the interpretation of her policy mix, and the significance of her media portrayal and international engagements. The evidence presented supports a view of Hanson as a prominent figure within the spectrum of right-wing populism in Australia, though the exact categorization remains debated.

In [17]:
display(Markdown(judgement))

# Claim

Australia's Pauline Hanson is a far right wing populist politician.

## Verdict

Mostly True

## Confidence

80%

## Strongest Evidence Supporting the Claim

- Hanson consistently promotes nationalist, anti-immigration, and anti-multiculturalism views, especially targeting Muslim immigrants, which aligns with far-right populism.  
- She uses a populist style positioning herself as a voice for "ordinary Australians" against elites, a classic populist narrative.  
- Her policy positions emphasize protectionism, law and order, skepticism of international institutions, and national sovereignty, consistent with global far-right populist movements.  
- Media, political analysts, and scholars widely describe her as far-right populist, reflecting a broad consensus.  
- She is part of a global surge in right-wing populism and engages with far-right figures internationally, reinforcing her alignment with far-right movements.  
- Her populism is inseparable from far-right ideological content, combining nationalism and cultural protectionism.

## Strongest Evidence Against the Claim

- Hanson’s political stance is complex and includes policies (e.g., support for nuclear energy) that do not fit neatly into far-right ideology, indicating eclecticism.  
- Populism is ideologically flexible and used across the political spectrum; thus, populism alone does not define her as far-right.  
- Media and scholarly labels may be subjective and influenced by political biases; consensus does not guarantee objective classification.  
- Hanson may better fit the category of "radical right populism," which contests but does not outright reject liberal democracy, rather than the more extreme far-right.  
- Engagements with far-right figures abroad may be strategic publicity rather than ideological alignment.

## Key Rebuttals

- Advocates argue that contemporary political analysis includes radical right populism within the broad far-right category, and Hanson’s nationalist and exclusionary rhetoric fits this definition.  
- Eclectic policies are common in far-right populism globally and do not negate far-right identity.  
- Consistent descriptions by reputable sources reflect informed consensus rather than mere stereotyping.  
- Engagements with far-right figures reinforce ideological alignment and are part of her political brand.  
- Critics caution that the far-right label risks conflating distinct political phenomena and that media framing can influence labeling.

## Unresolved Issues

- The precise boundary between "far-right" and "radical right populist" remains debated.  
- The extent to which media and scholarly labeling objectively capture Hanson’s political identity versus reflecting political context or bias.  
- Whether her international associations are ideological or strategic.  
- The significance of her eclectic policy positions in defining her as far-right.

## Reasoning

The evidence strongly supports that Pauline Hanson exhibits key characteristics of far-right populism, including nationalist, anti-immigration rhetoric, populist style, and policy positions aligned with global far-right movements. The broad media and scholarly consensus further reinforces this classification. However, the presence of some non-traditional far-right policies and the ideological flexibility of populism introduce nuance, suggesting that while she fits within the far-right populist spectrum, her political identity is complex and not purely extremist. The debate acknowledges these nuances but does not provide sufficient evidence to exclude her from the far-right populist category.

## Final Assessment

Based on the debate summary, Pauline Hanson is best described as a far right wing populist politician in Australia, though with some qualifications due to her eclectic policies and the flexible nature of populism. The claim is mostly true given the strong alignment of her rhetoric, style, policies, and public perception with far-right populism.

# Pauline Hanson's One Nation Policies

Pauline Hanson is a far right wing populist politician. https://en.wikipedia.org/wiki/Pauline_Hanson

We will debate each of her political policies, one by one. 

A neutral summary of each debate topic will be given at the end of each full transcript.

In [18]:
policy_urls = [
    "https://www.onenation.org.au/oil-and-gas",
    "https://www.onenation.org.au/famly-tax-policy",
    "https://www.onenation.org.au/slash-waste",
    "https://www.onenation.org.au/free-speech-policy",
    "https://www.onenation.org.au/reduce-cost-of-living",
    "https://www.onenation.org.au/citizen-initiated-referenda",
    "https://www.onenation.org.au/climate",
    "https://www.onenation.org.au/family",
    "https://www.onenation.org.au/farming",
    "https://www.onenation.org.au/foreign-ownership",
    "https://www.onenation.org.au/education",
    "https://www.onenation.org.au/immigration",
    "https://www.onenation.org.au/cannabis",
    "https://www.onenation.org.au/rorts",
    "https://www.onenation.org.au/tax",
    "https://www.onenation.org.au/firearms",
    "https://www.onenation.org.au/health",
    "https://www.onenation.org.au/covid",
    "https://www.onenation.org.au/pro-life",
    "https://www.onenation.org.au/fisheries",
    "https://www.onenation.org.au/forestry",
    "https://www.onenation.org.au/real-impacts-of-net-zero-australia",
    "https://www.onenation.org.au/net-zero-destroying-australia",
    "https://www.onenation.org.au/embracing-australian-coal-reliable-energy-future",
    "https://www.onenation.org.au/embracing-nuclear-energy-hydro-australia-future",
    "https://www.onenation.org.au/lowering-prices-households-business-industry-australia"
]

In [ ]:
from pathlib import Path
import re
from time import time

def slugify(text: str, max_length: int = 80) -> str:
    """Convert text into a safe filename."""
    slug = text.lower().strip()
    slug = re.sub(r"[^\w\s-]", "", slug)
    slug = re.sub(r"[-\s]+", "-", slug)
    return slug[:max_length].strip("-")

def debate_by_policy_urls_and_save_to_file(policy_urls):
    output_dir = Path(f"ai-debates-V2-{MODEL}-{int(time())}")
    output_dir.mkdir(parents=True, exist_ok=True)

    for index, policy_url in enumerate(policy_urls, start=1):
        try:
            # --------------------------------------------------
            # Fetch policy
            # --------------------------------------------------

            policy_content = fetch_website_contents(policy_url)

            # --------------------------------------------------
            # Convert policy into a debatable claim
            # --------------------------------------------------

            claim = convert_policy_to_claim(policy_content)

            print(f"\n{'=' * 80}")
            print(f"Debate {index}: {claim}")
            print(f"{'=' * 80}")

            # --------------------------------------------------
            # Run debate
            # --------------------------------------------------

            transcript = debate(
                claim=claim,
                rounds=DEFAULT_ROUNDS,
            )

            # --------------------------------------------------
            # Summarise debate
            # --------------------------------------------------

            summary = summarise_debate(
                claim=claim,
                transcript=transcript,
            )

            # --------------------------------------------------
            # Judge debate
            # --------------------------------------------------
            judgement = judge_summary(
                claim=claim,
                summary=summary,
            )

            # --------------------------------------------------
            # Write debate to Markdown
            # --------------------------------------------------

            filename = f"{index:03d}-{slugify(claim)}.md"
            output_path = output_dir / filename

            markdown = f"""# A Multi Agent AI Policy Debate
AI can make mistakes. Always critically evaluate the content of the debate.


# Claim

{claim}


# Source

{policy_url}


# Debate

{transcript_to_markdown(transcript)}


# Summary

{summary}


# Judgement

{judgement}
"""

            output_path.write_text(
                markdown,
                encoding="utf-8",
            )

            print(f"Saved: {output_path}")

        except Exception as exc:
            print(f"ERROR processing {policy_url}: {exc}")

In [20]:
# Run the debate and save the results to files
debate_by_policy_urls_and_save_to_file(policy_urls)

The Australian government should implement a joint venture scheme granting public ownership of up to 30% in oil and gas projects and replace the Petroleum Resource Rent Tax with a simple royalty on all extracted resources.

Debate 1: The Australian government should implement a joint venture scheme granting public ownership of up to 30% in oil and gas projects and replace the Petroleum Resource Rent Tax with a simple royalty on all extracted resources.

CLAIM
The Australian government should implement a joint venture scheme granting public ownership of up to 30% in oil and gas projects and replace the Petroleum Resource Rent Tax with a simple royalty on all extracted resources.

ADVOCATE — OPENING STATEMENT
The Australian government should implement a joint venture scheme granting public ownership of up to 30% in oil and gas projects and replace the Petroleum Resource Rent Tax (PRRT) with a simple royalty on all extracted resources for several compelling reasons:

1. **Ensuring Fair Pu

KeyboardInterrupt: 